In [1]:
import litellm
import json

# A sample scenario from our RAG pipeline output
eval_test_case = {
    "query": "What network protocol does MeshQuery use?",
    "retrieved_context": "MeshQuery routes requests across cluster nodes using an ultra-low latency RSocket protocol layer over TCP.",
    "generated_answer": "MeshQuery handles communication using the RSocket protocol. It also securely encrypts all data packets using enterprise-grade AES-256 bits standards."
}

print("✓ Cell 1: Evaluation test case loaded. Notice that the generated answer mentions AES-256, which is NOT in the context!")

14:30:43 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
14:30:44 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


✓ Cell 1: Evaluation test case loaded. Notice that the generated answer mentions AES-256, which is NOT in the context!


In [6]:
def calculate_faithfulness(context: str, answer: str) -> float:
    # Prompt 1: Extract separate factual claims from the answer
    extraction_prompt = [
        {
            "role": "system",
            "content": (
                "Your job is to break down the following text into an array of individual, distinct factual claims.\n"
                "Output your response ONLY as a valid JSON array of strings, like this: ['claim 1', 'claim 2']."
            )
        },
        {
            "role": "user",
            "content": f"Text: {answer}"
        }
    ]
    
    response = litellm.completion(
        model="ollama/llama3.2",
        messages=extraction_prompt,
        api_base="http://localhost:11434",
        temperature=0.0
    )
    
    claims = json.loads(response.choices[0].message.content.strip())
    print(f"\nExtracted Factual Claims to Verify: {claims}")
    
    verified_count = 0
    
    # Prompt 2: Verify each claim against the retrieved context block
    for claim in claims:
        verification_prompt = [
            {
                "role": "system",
                "content": (
                    f"Context: {context}\n\n"
                    "Analyze if the user's statement is explicitly supported by the context above.\n"
                    "Respond with exactly one character: 'Y' if it is supported, or 'N' if it is not supported. Do not explain."
                )
            },
            {
                "role": "user",
                "content": f"Statement: {claim}"
            }
        ]
        
        v_res = litellm.completion(
            model="ollama/llama3.2",
            messages=verification_prompt,
            api_base="http://localhost:11434",
            temperature=0.0
        )
        
        verdict = v_res.choices[0].message.content.strip().upper()
        print(f"  - Claim: '{claim}' | Verified by Context? [{verdict}]")
        if 'Y' in verdict:
            verified_count += 1
            
    # Compute final metric fraction
    score = verified_count / len(claims) if claims else 0.0
    return score

print("✓ Cell 2: Faithfulness evaluation algorithm successfully compiled.")

✓ Cell 2: Faithfulness evaluation algorithm successfully compiled.


In [3]:
def calculate_answer_relevance(query: str, answer: str) -> float:
    prompt = [
        {
            "role": "system",
            "content": (
                "You are an independent quality control judge. Rate whether the provided answer "
                "directly addresses and answers the user's question, or if it is off-topic/fluff.\n"
                "Provide a score from 1 to 5, where 1 is completely irrelevant and 5 is perfectly relevant.\n"
                "Output ONLY the raw integer digit score. Do not include words or explanations."
            )
        },
        {
            "role": "user",
            "content": f"Question: {query}\nAnswer: {answer}"
        }
    ]
    
    response = litellm.completion(
        model="ollama/llama3.2",
        messages=prompt,
        api_base="http://localhost:11434",
        temperature=0.0
    )
    
    try:
        raw_score = int(response.choices[0].message.content.strip())
        # Normalize 1-5 score down to a standard 0.0 - 1.0 float
        normalized_score = (raw_score - 1) / 4.0
    except Exception:
        normalized_score = 0.0
        
    return normalized_score

print("✓ Cell 3: Answer relevance evaluation metric compiled.")

✓ Cell 3: Answer relevance evaluation metric compiled.


In [8]:
# 1. Calculate metrics
faithfulness_score = calculate_faithfulness(
    context=eval_test_case["retrieved_context"],
    answer=eval_test_case["generated_answer"]
)

relevance_score = calculate_answer_relevance(
    query=eval_test_case["query"],
    answer=eval_test_case["generated_answer"]
)

# 2. Print output dashboard report
print("\n===========================================")
print("     RAG PRODUCTION EVALUATION REPORT      ")
print("===========================================")
print(f"FAITHFULNESS SCORE   : {faithfulness_score:.2f} / 1.00")
print(f"ANSWER RELEVANCE SCORE: {relevance_score:.2f} / 1.00")
print("===========================================")
if faithfulness_score < 1.0:
    print("🚨 WARNING: Detected hallucinated facts not backed by source documents!")
else:
    print("✅ PASS: Answer is completely grounded in retrieved context.")


Extracted Factual Claims to Verify: ['MeshQuery uses the RSocket protocol for communication', 'MeshQuery securely encrypts all data packets using AES-256 bits standards']
  - Claim: 'MeshQuery uses the RSocket protocol for communication' | Verified by Context? [Y]
  - Claim: 'MeshQuery securely encrypts all data packets using AES-256 bits standards' | Verified by Context? [N]

     RAG PRODUCTION EVALUATION REPORT      
FAITHFULNESS SCORE   : 0.50 / 1.00
ANSWER RELEVANCE SCORE: 0.75 / 1.00
🚨 WARNING: Detected hallucinated facts not backed by source documents!
